<a href="https://colab.research.google.com/github/Zain506/MedCLIP-SAM/blob/main/notebooks/ImageSegmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SAM


[MedCLIP-SAM](https://arxiv.org/pdf/2403.20253)
> With a fine-tuned BiomedCLIP model, we proposed a zero-shot universal medical image segmentation strategy, which leverages the recent XAI technique,
gScoreCAM that provides visual saliency maps of text prompts in corresponding images for CLIP models. While gScoreCAM was shown to outperform
gradCAM in natural images in accuracy and specificity, we adopted it in radiological tasks for the first time. Here, for an input image and a text prompt for
the target anatomy/pathology, we first obtained an initial, coarse segmentation
by post-processing the gScoreCAM map with a conditional random field (CRF)
filter, which was then used to obtain a bounding box for SAM to produce a
pseudo-mask as zero-shot segmentation. In the attempt to further enhance the
accuracy of zero-shot segmentation, we used the resulting pseudo-masks to train
a Residual UNet in a weakly supervised setting.

[gScoreCAM](https://www.google.com/url?q=https%3A%2F%2Fopenaccess.thecvf.com%2Fcontent%2FACCV2022%2Fpapers%2FChen_gScoreCAM_What_objects_is_CLIP_looking_at_ACCV_2022_paper.pdf)

## Summary

1. Pass in an image and text datapoint
2. Feed to BiomedCLIP post fine-tuning
3. Feed embedding to gScoreCAM
4. Apply conditional random field (CRF) filter to gScoreCAM output to get bounding box for zero-shot segmentation.
5. Apply bounding box to SAM


gScoreCAM feeds an image through the encoder to obtain all of the convolutions. It then uses these as a "mask" (Hadamaard product with original image)

Then you encode each masked image again and calculate the similarity to the text embedding (to see which mask fits best)

Then the final mask is a weighted sum with the similarity score of each mask.

In [33]:
%pip install open-clip-torch -q

In [ ]:
# Import data
from datasets import load_dataset
ds = load_dataset("adishourya/MEDPIX-ClinQA") # Same MedPIX dataset that fine-tuned MedCLIP
train_valid = ds["train"].train_test_split(test_size=0.1)
training = train_valid["train"].select(range(100))
test = train_valid["test"]

In [ ]:
print(training)

Dataset({
    features: ['image_id', 'mode', 'case_id', 'question', 'answer'],
    num_rows: 100
})


In [18]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Load BiomedCLIP

In [19]:
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
from PIL import Image
import open_clip
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
weights_path = "/content/drive/MyDrive/biomedclip_weights.pth"
model, _, preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="laion2b_s34b_b79k")
state_dict = torch.load(weights_path, map_location=device)
model.load_state_dict(state_dict)
tokenizer = open_clip.get_tokenizer("ViT-B-32")

In [29]:
def collate_fn(batch): # Convert batch into tensor
  images = torch.stack([preprocess(x["image_id"]) for x in batch])
  texts = tokenizer([x["answer"] for x in batch])
  return images, texts
data_loader = DataLoader(
    training,
    batch_size=10,
    shuffle=True,
    collate_fn=collate_fn,
)


In [ ]:
epochs = 1
for epoch in range(epochs):
  for images, texts in tqdm(data_loader, desc=f"Epoch {epoch+1}"):
    images = images.to(device)
    texts = texts.to(device)
    I = model.encode_image(images)
    T = model.encode_text(texts)
    break

In [31]:
print(I.shape)

torch.Size([10, 512])
